# CLIP ViT-L/14 features

the same as `clip_features.ipynb` with the larger encoder: every face crop becomes 768 numbers with a frozen CLIP ViT-L/14 (OpenAI weights). same crops, same manifests and same row order, so every later notebook only changes its feature path.

ViT-L/14 is about 4.5 times the compute of ViT-B/16, so on one T4 this takes about 3 hours. with the T4 x2 accelerator both gpus are used and it should take about half that.

In [1]:
config = {
    "limit": 0,
    # the OpenAI weights were trained with QuickGELU
    "model": "ViT-L-14-quickgelu",
    "pretrained": "openai",
    "batch_size": 128,
    "workers": 4,
    "out": "/kaggle/working/features_l14",
}
parts = ["ffpp/train", "ffpp/val", "ffpp/test", "celebdf/test"]

In [2]:
!pip install -q open_clip_torch==3.3.0

In [3]:
import os
import glob
import json
import time

import numpy as np
import pandas as pd
import torch
from PIL import Image
import open_clip

In [4]:
def find(pattern):
    for depth in range(8):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + pattern)
        if len(hits) > 0:
            return hits[0]
    raise FileNotFoundError(pattern)


class Crops(torch.utils.data.Dataset):
    def __init__(self, paths, preprocess):
        self.paths = paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.preprocess(Image.open(self.paths[i]).convert("RGB"))

In [5]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
model, _, preprocess = open_clip.create_model_and_transforms(config["model"], pretrained=config["pretrained"])
model = model.to(device).eval()
# only the image side is needed, split over both gpus when there are two
visual = model.visual
if torch.cuda.device_count() > 1:
    visual = torch.nn.DataParallel(visual)
print("gpus:", torch.cuda.device_count(), " output size:", model.visual.output_dim)

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

gpus: 2  output size: 768


In [6]:
os.makedirs(config["out"], exist_ok=True)
for part in parts:
    t0 = time.time()
    name = part.replace("/", "_")
    rows = pd.read_csv(find(name + "_manifest.csv"), dtype={"video_id": str})
    root = find(part)[:-len(part)]
    if config["limit"] > 0:
        videos = rows[["method", "video_id"]].drop_duplicates().head(config["limit"])
        rows = rows.merge(videos)
    paths = [os.path.join(root, p) for p in rows["path"]]
    missing = [p for p in paths if not os.path.exists(p)]
    assert len(missing) == 0, missing[:3]

    loader = torch.utils.data.DataLoader(Crops(paths, preprocess), batch_size=config["batch_size"],
                                         num_workers=config["workers"])
    chunks = []
    with torch.no_grad(), torch.autocast("cuda"):
        for batch in loader:
            chunks.append(visual(batch.to(device)).float().cpu().numpy())
    features = np.concatenate(chunks)

    np.save(os.path.join(config["out"], name + "_clip.npy"), features)
    rows.to_csv(os.path.join(config["out"], name + "_rows.csv"), index=False)
    print(name, features.shape, "mean norm %.2f" % np.linalg.norm(features, axis=1).mean(), "%.0f min" % ((time.time() - t0) / 60))

with open(os.path.join(config["out"], "encoder.json"), "w") as f:
    json.dump({"model": config["model"], "pretrained": config["pretrained"], "open_clip": open_clip.__version__}, f, indent=1)

ffpp_train (46076, 768) mean norm 18.18 42 min
ffpp_val (22396, 768) mean norm 18.11 18 min
ffpp_test (22400, 768) mean norm 18.19 19 min
celebdf_test (16572, 768) mean norm 18.87 12 min
